# ggml-python in Pyodide

This notebook installs the Pyodide wheel and runs a small ggml graph in the browser.

In [ ]:
import micropip
import piplite

await micropip.install(["numpy", "typing_extensions"])
await piplite.install(
    "ggml-python",
    deps=False,
)

In [ ]:
import ggml

params = ggml.ggml_init_params(mem_size=16 * 1024 * 1024, mem_buffer=None)
ctx = ggml.ggml_init(params)
assert ctx is not None

x = ggml.ggml_new_tensor_1d(ctx, ggml.GGML_TYPE_F32, 1)
a = ggml.ggml_new_tensor_1d(ctx, ggml.GGML_TYPE_F32, 1)
b = ggml.ggml_new_tensor_1d(ctx, ggml.GGML_TYPE_F32, 1)

x2 = ggml.ggml_mul(ctx, x, x)
f = ggml.ggml_add(ctx, ggml.ggml_mul(ctx, a, x2), b)

gf = ggml.ggml_new_graph(ctx)
ggml.ggml_build_forward_expand(gf, f)

ggml.ggml_set_f32(x, 2.0)
ggml.ggml_set_f32(a, 3.0)
ggml.ggml_set_f32(b, 4.0)

ggml.ggml_graph_compute_with_ctx(ctx, gf, 1)
output = ggml.ggml_get_f32_1d(f, 0)
ggml.ggml_free(ctx)

output

In [ ]:
assert output == 16.0
ggml.ggml_version().decode()